In [2]:
import queue
import time
import random

# Configuration for each machine
machine_config = {
    1: {'processing_time': 3, 'batch_size': 5},
    2: {'processing_time': 2, 'setup_time': {'1to2': 2, '2to1': 1}},
    3: {'processing_time': 4, 'batch_size': 5},
    4: {'processing_time': 1, 'setup_time': {'1to2': 1, '2to1': 2}}
}

# Queues for each machine
queues = [queue.Queue() for _ in range(4)]

# Simulation parameters
SIMULATION_TIME = 60  # Total simulation time in seconds
PRODUCT_ARRIVAL_RATE = 5  # New product every ~5 seconds
LOG_INTERVAL = 5  # Log output every 5 time units

# Helper function to process a batch in batch processing machines (1 and 3)
def process_batch(machine_index):
    batch_size = machine_config[machine_index + 1]['batch_size']
    if queues[machine_index].qsize() >= batch_size:
        batch = [queues[machine_index].get() for _ in range(batch_size)]
        time.sleep(machine_config[machine_index + 1]['processing_time'])
        return f"Processed batch of products {batch}"
    return None

# Helper function for machines with setup times (2 and 4)
def process_with_setup(machine_index, last_product):
    if not queues[machine_index].empty():
        product = queues[machine_index].queue[0]  # Peek without removing
        setup_key = f"{last_product}to{product}"
        if last_product and product != last_product and setup_key in machine_config[machine_index + 1]['setup_time']:
            time.sleep(machine_config[machine_index + 1]['setup_time'][setup_key])
            queues[machine_index].get()  # Remove the product from queue after setup
            return f"Setup from product {last_product} to {product}"
        time.sleep(machine_config[machine_index + 1]['processing_time'])
        queues[machine_index].get()  # Remove the product from queue after processing
        return f"Processed product {product}"
    return None

# Main simulation loop
def simulate():
    last_products = [None, None]  # Last processed products for machines 2 and 4
    start_time = time.time()
    current_time = start_time
    next_arrival_time = 0
    next_log_time = LOG_INTERVAL

    while current_time - start_time < SIMULATION_TIME:
        # Simulate product arrival at random intervals
        if current_time >= next_arrival_time:
            product = random.choice([1, 2])
            queues[0].put(product)
            next_arrival_time = current_time + random.expovariate(1.0 / PRODUCT_ARRIVAL_RATE)

        # Process each machine
        status = [None] * 4
        status[0] = process_batch(0)  # Machine 1
        if status[0]:
            for _ in range(machine_config[1]['batch_size']):
                queues[1].put(product)

        status[1] = process_with_setup(1, last_products[0])  # Machine 2
        if status[1] and "Processed" in status[1]:
            queues[2].put(product)
            last_products[0] = product

        status[2] = process_batch(2)  # Machine 3
        if status[2]:
            for _ in range(machine_config[3]['batch_size']):
                queues[3].put(product)

        status[3] = process_with_setup(3, last_products[1])  # Machine 4
        if status[3] and "Processed" in status[3]:
            last_products[1] = product

        # Log output every 5 seconds
        if current_time - start_time >= next_log_time:
            print(f"Time: {current_time - start_time:.2f}s")
            for i in range(4):
                print(f"Machine {i+1} queue: {[item for item in list(queues[i].queue)]}")
                print(f"Machine {i+1} status: {status[i] if status[i] else 'Idle'}")
            next_log_time += LOG_INTERVAL

        # Increment time
        current_time = time.time()

# Run the simulation
simulate()


Time: 5.00s
Machine 1 queue: [1, 1, 2, 2]
Machine 1 status: Idle
Machine 2 queue: []
Machine 2 status: Idle
Machine 3 queue: []
Machine 3 status: Idle
Machine 4 queue: []
Machine 4 status: Idle
Time: 13.86s
Machine 1 queue: [1]
Machine 1 status: Idle
Machine 2 queue: [2, 2, 2]
Machine 2 status: Processed product 2
Machine 3 queue: [2, 1]
Machine 3 status: Idle
Machine 4 queue: []
Machine 4 status: Idle
Time: 15.86s
Machine 1 queue: [1, 2]
Machine 1 status: Idle
Machine 2 queue: [2, 2]
Machine 2 status: Setup from product 1 to 2
Machine 3 queue: [2, 1]
Machine 3 status: Idle
Machine 4 queue: []
Machine 4 status: Idle
Time: 21.86s
Machine 1 queue: [1, 2, 2]
Machine 1 status: Idle
Machine 2 queue: []
Machine 2 status: Idle
Machine 3 queue: [2, 1]
Machine 3 status: Idle
Machine 4 queue: []
Machine 4 status: Idle
Time: 25.00s
Machine 1 queue: [1, 2, 2]
Machine 1 status: Idle
Machine 2 queue: []
Machine 2 status: Idle
Machine 3 queue: [2, 1]
Machine 3 status: Idle
Machine 4 queue: []
Machine

In [ ]:
#Next steps
# 1) Introduce a demand forecast into the state space and reward it for meeting future excess demand
# 2) See if it can switch demand patterns quickly 
# 3) Q-learning implementaion to achive the same result
# 4) Write a LP program of this